## 1. Импорты и конфигурация

In [ ]:
import asyncio
import json
import random
import re
import time
from collections import Counter, defaultdict
from pathlib import Path
from typing import Literal

import dspy
import pandas as pd
from datasets import load_dataset
from openai import AsyncOpenAI, OpenAI
from pydantic import BaseModel, Field, ValidationError
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

SEED = 42
SAMPLE_SIZE = 200
TRAIN_SIZE = 20
DEV_SIZE = 40
TEST_SIZE = SAMPLE_SIZE - TRAIN_SIZE - DEV_SIZE
ENTITY_TYPES = ["PER", "LOC", "ORG", "MISC"]

PRIMARY_BASE_URL = "http://127.0.0.1:8080/v1"
PRIMARY_MODEL_ID = None 
API_KEY = "local"

SECOND_BASE_URL = "http://127.0.0.1:8081/v1"
SECOND_MODEL_ID = None
RUN_SECOND_MODEL = True

RUN_DSPY_OPTIMIZATION = True
RUN_ORDERED_DSPY_OPTIMIZATION = False
DSPY_AUTO_MODE = "light"
MAX_CONCURRENCY = 2

OUTPUT_DIR = Path("ner_outputs")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
CACHE_DIR = Path("ner_cache")
CACHE_DIR.mkdir(exist_ok=True, parents=True)

print("DSPy:", getattr(dspy, "__version__", "unknown"))
print("Размер выборки:", SAMPLE_SIZE, "| train/dev/test:", TRAIN_SIZE, DEV_SIZE, TEST_SIZE)

/Users/veronika_steklo/projects/ml_ssu/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DSPy: 3.4.0
Размер выборки: 200 | train/dev/test: 20 40 140


## 2. Подключаемся к основной локальной модели через OpenAI SDK

In [ ]:
primary_client = OpenAI(base_url=PRIMARY_BASE_URL, api_key=API_KEY, timeout=120.0)
primary_async_client = AsyncOpenAI(base_url=PRIMARY_BASE_URL, api_key=API_KEY, timeout=180.0)

primary_models = primary_client.models.list()
assert primary_models.data, f"Нет моделей на {PRIMARY_BASE_URL}. Проверь, что llama-server запущен."
PRIMARY_MODEL_ID = PRIMARY_MODEL_ID or primary_models.data[0].id
print("Primary endpoint:", PRIMARY_BASE_URL)
print("Primary model ID:", PRIMARY_MODEL_ID)
MODEL_SLUG = re.sub(r"[^A-Za-z0-9._-]+", "_", PRIMARY_MODEL_ID).strip("_") or "local_model"

smoke = primary_client.chat.completions.create(
    model=PRIMARY_MODEL_ID,
    messages=[{"role": "user", "content": "Reply with exactly: LOCAL_OK"}],
    temperature=0,
    max_tokens=20,
)
print("Проверка сервера:", smoke.choices[0].message.content)

Primary endpoint: http://127.0.0.1:8080/v1
Primary model ID: /Users/veronika_steklo/Models/qwen2.5-7b-instruct-q3_k_m.gguf
Проверка сервера: LOCAL_OK


## 3. Загружаем CoNLL-2003 и выбираем 200 примеров из train

In [3]:
dataset = load_dataset("tner/conll2003", revision="refs/convert/parquet")
train_data = dataset["train"]

id2tag = {
    0: "O", 1: "B-ORG", 2: "B-MISC", 3: "B-PER", 4: "I-PER",
    5: "B-LOC", 6: "I-ORG", 7: "I-MISC", 8: "I-LOC",
}

random.seed(SEED)
selected_indices = sorted(random.sample(range(len(train_data)), SAMPLE_SIZE))
sentences = [train_data[i]["tokens"] for i in selected_indices]
gold_tags = [[id2tag[tag] for tag in train_data[i]["tags"]] for i in selected_indices]
assert all(len(x) == len(y) for x, y in zip(sentences, gold_tags))

print(dataset)
print("Пример предложения:", " ".join(sentences[0]))
print("BIO-теги:", gold_tags[0])

DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 3453
    })
})
Пример предложения: Seat posted 3,420 registrations compared with 5522 registrations in July a year earlier .
BIO-теги: ['B-ORG', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


## 4. Преобразуем BIO-разметку в список эталонных сущностей

In [4]:
def bio_to_entities(tokens, tags):
    """BIO -> список словарей {text, type, start, end}; end не включается."""
    entities = []
    current_tokens = []
    current_type = None
    start = None

    def flush(end):
        nonlocal current_tokens, current_type, start
        if current_tokens:
            entities.append({
                "text": " ".join(current_tokens),
                "type": current_type,
                "start": start,
                "end": end,
            })
        current_tokens = []
        current_type = None
        start = None

    for i, (token, tag) in enumerate(zip(tokens, tags)):
        if tag == "O":
            flush(i)
            continue
        prefix, entity_type = tag.split("-", 1)
        if prefix == "B" or entity_type != current_type:
            flush(i)
            current_tokens = [token]
            current_type = entity_type
            start = i
        else:
            current_tokens.append(token)
    flush(len(tokens))
    return entities


gold_entities = [bio_to_entities(tokens, tags) for tokens, tags in zip(sentences, gold_tags)]

def group_gold(entities):
    grouped = {t: [] for t in ENTITY_TYPES}
    for entity in entities:
        grouped[entity["type"]].append(entity["text"])
    return grouped

print("Эталон для примера:")
print(json.dumps(gold_entities[0], ensure_ascii=False, indent=2))
print("Всего gold-упоминаний:", sum(len(x) for x in gold_entities))

Эталон для примера:
[
  {
    "text": "Seat",
    "type": "ORG",
    "start": 0,
    "end": 1
  }
]
Всего gold-упоминаний: 353


## 5. Разделяем данные до оптимизации

In [5]:
positions = list(range(SAMPLE_SIZE))
random.Random(SEED).shuffle(positions)
train_positions = positions[:TRAIN_SIZE]
dev_positions = positions[TRAIN_SIZE:TRAIN_SIZE + DEV_SIZE]
test_positions = positions[TRAIN_SIZE + DEV_SIZE:]
assert not (set(train_positions) & set(dev_positions) or set(train_positions) & set(test_positions) or set(dev_positions) & set(test_positions))

print(f"DSPy train: {len(train_positions)}")
print(f"DSPy validation: {len(dev_positions)}")
print(f"Final test: {len(test_positions)}")
print("Test indexes:", test_positions[:10], "...")

DSPy train: 20
DSPy validation: 40
Final test: 140
Test indexes: [102, 38, 156, 176, 135, 78, 115, 19, 14, 177] ...


## 6. Pydantic-схемы для двух форматов Structured Output

In [6]:
class EntitiesByType(BaseModel):
    PER: list[str] = Field(default_factory=list, description="Person names")
    LOC: list[str] = Field(default_factory=list, description="Geographical locations")
    ORG: list[str] = Field(default_factory=list, description="Organizations")
    MISC: list[str] = Field(default_factory=list, description="Other named entities")

class GroupedNERResult(BaseModel):
    entities: EntitiesByType

class EntityMention(BaseModel):
    text: str = Field(description="Exact entity text copied from the input sentence")
    type: Literal["PER", "LOC", "ORG", "MISC"]

class OrderedNERResult(BaseModel):
    entities: list[EntityMention] = Field(
        default_factory=list,
        description="Entities in order of appearance in the input sentence"
    )

print("Grouped JSON schema keys:", GroupedNERResult.model_json_schema()["properties"].keys())
print("Ordered JSON schema keys:", OrderedNERResult.model_json_schema()["properties"].keys())

Grouped JSON schema keys: dict_keys(['entities'])
Ordered JSON schema keys: dict_keys(['entities'])


## 7. Промпты: исходный и улучшенный вручную

In [7]:
PROMPT_V1 = """You are a named entity recognition (NER) system.
Extract all named entities explicitly mentioned in the input sentence.
Classify each entity into exactly one category:
- PER: person names
- LOC: geographical locations
- ORG: organizations
- MISC: other named entities, such as nationalities, languages, events,
  products, and named works when appropriate.
Rules:
1. Extract only entities explicitly present in the sentence.
2. Preserve the original spelling and wording of each entity.
3. Do not include ordinary words that are not named entities.
4. Assign each entity the most appropriate category.
5. Do not omit entities just because their type is ambiguous.
6. If a category has no entities, return an empty list.
"""

PROMPT_V2 = """You are a precise NER system following the CoNLL-2003 entity scheme.
Extract only entity spans explicitly present in the input sentence.
Use exactly these types:
- PER: person names only.
- LOC: countries, cities, regions, and other geographical places.
- ORG: companies, institutions, organizations, sports teams, and media organizations.
- MISC: named entities outside PER/LOC/ORG, especially nationalities, ethnic,
  religious or political groups, languages, and events when they function as names.

Strict rules:
1. Copy the exact words from the input; do not translate, normalize, expand, or paraphrase them.
2. Do not replace an adjective/nationality with the corresponding country name:
   "Syrian" is not "Syria".
3. For PER, exclude titles and occupations (e.g. President, coach, minister) unless
   the word is actually part of the person's name.
4. For ORG, include the complete organization/team name, but do not absorb nearby
   locations or ordinary descriptive words.
5. Do not classify dates, years, scores, numbers, generic nouns, job titles,
   or ordinary adjectives as entities merely because they look important.
6. Do not extract dataset markup such as -LRB-, -RRB-, -LRP-, -RRP-, -LPR-, -RPR-.
7. Do not invent missing entities or add information based on world knowledge.
8. Each mention must have exactly one type. Never duplicate a mention across types.
9. Preserve distinct entities separately; do not merge neighboring entities.
10. Return empty lists for categories with no entities.
"""

## 8. Универсальный вызов Structured Output через OpenAI SDK + Pydantic

In [ ]:
def extract_json_from_text(text):
    if not isinstance(text, str):
        raise ValueError("Model returned no text")
    cleaned = text.strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned, flags=re.I)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    start_obj, end_obj = cleaned.find("{"), cleaned.rfind("}")
    start_arr, end_arr = cleaned.find("["), cleaned.rfind("]")
    candidates = []
    if start_obj >= 0 and end_obj > start_obj:
        candidates.append(cleaned[start_obj:end_obj + 1])
    if start_arr >= 0 and end_arr > start_arr:
        candidates.append(cleaned[start_arr:end_arr + 1])
    for candidate in candidates:
        try:
            return json.loads(candidate)
        except json.JSONDecodeError:
            pass
    raise ValueError(f"No valid JSON found in response: {cleaned[:250]!r}")


async def call_structured(client, model_id, sentence, system_prompt, schema, max_tokens=512):
    """Use OpenAI SDK parse/Pydantic first; fallback to JSON mode + Pydantic validation."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": "Extract named entities from this sentence:\n" + sentence},
    ]
    start = time.perf_counter()
    method = "pydantic_parse"
    try:
        response = await client.chat.completions.parse(
            model=model_id,
            messages=messages,
            response_format=schema,
            temperature=0,
            max_tokens=max_tokens,
        )
        message = response.choices[0].message
        if message.refusal:
            raise RuntimeError(f"Model refusal: {message.refusal}")
        if message.parsed is not None:
            parsed = message.parsed
        else:
            parsed = schema.model_validate(extract_json_from_text(message.content or ""))
        usage = getattr(response, "usage", None)
        return parsed, {
            "elapsed_seconds": time.perf_counter() - start,
            "method": method,
            "prompt_tokens": getattr(usage, "prompt_tokens", None),
            "completion_tokens": getattr(usage, "completion_tokens", None),
        }
    except Exception as parse_error:
        method = "json_mode_fallback"
        response = await client.chat.completions.create(
            model=model_id,
            messages=messages + [{
                "role": "system",
                "content": "Return only one valid JSON object matching the requested schema. No markdown.",
            }],
            response_format={"type": "json_object"},
            temperature=0,
            max_tokens=max_tokens,
        )
        message = response.choices[0].message
        parsed = schema.model_validate(extract_json_from_text(message.content or ""))
        usage = getattr(response, "usage", None)
        return parsed, {
            "elapsed_seconds": time.perf_counter() - start,
            "method": method,
            "parse_error": repr(parse_error),
            "prompt_tokens": getattr(usage, "prompt_tokens", None),
            "completion_tokens": getattr(usage, "completion_tokens", None),
        }


async def run_structured_batch(client, model_id, tokens_batch, prompt, schema, concurrency=1, cache_path=None):
    results = [None] * len(tokens_batch)
    metadata = [None] * len(tokens_batch)
    errors = []
    semaphore = asyncio.Semaphore(concurrency)

    async def one(i, tokens):
        async with semaphore:
            try:
                parsed, info = await call_structured(
                    client, model_id, " ".join(tokens), prompt, schema
                )
                results[i] = parsed.model_dump()
                metadata[i] = info
            except Exception as exc:
                errors.append({"local_index": i, "sentence": " ".join(tokens), "error": repr(exc)})

    start = time.perf_counter()
    await asyncio.gather(*(one(i, tokens) for i, tokens in enumerate(tokens_batch)))
    elapsed = time.perf_counter() - start
    output = {"predictions": results, "metadata": metadata, "errors": errors, "elapsed_seconds": elapsed}
    if cache_path:
        Path(cache_path).write_text(json.dumps(output, ensure_ascii=False, indent=2), encoding="utf-8")
    return output

## 9. Запускаем исходный промпт и улучшенный ручной промпт на test

In [ ]:
test_tokens = [sentences[i] for i in test_positions]
cache_v1 = CACHE_DIR / f"{MODEL_SLUG}_grouped_prompt_v1.json"
cache_v2 = CACHE_DIR / f"{MODEL_SLUG}_grouped_prompt_v2.json"

async def load_or_run(path, client, model_id, tokens, prompt, schema):
    if path.exists():
        print("Кеш найден:", path)
        return json.loads(path.read_text(encoding="utf-8"))
    print("Запускаю модель; это может занять несколько минут:", path.name)
    return await run_structured_batch(client, model_id, tokens, prompt, schema,
                                      concurrency=MAX_CONCURRENCY, cache_path=path)

v1_result = await load_or_run(cache_v1, primary_async_client, PRIMARY_MODEL_ID,
                              test_tokens, PROMPT_V1, GroupedNERResult)
v2_result = await load_or_run(cache_v2, primary_async_client, PRIMARY_MODEL_ID,
                              test_tokens, PROMPT_V2, GroupedNERResult)

print("V1 elapsed:", round(v1_result["elapsed_seconds"], 2), "sec; errors:", len(v1_result["errors"]))
print("V2 elapsed:", round(v2_result["elapsed_seconds"], 2), "sec; errors:", len(v2_result["errors"]))

Кеш найден: ner_cache/Users_veronika_steklo_Models_qwen2.5-7b-instruct-q3_k_m.gguf_grouped_prompt_v1.json
Кеш найден: ner_cache/Users_veronika_steklo_Models_qwen2.5-7b-instruct-q3_k_m.gguf_grouped_prompt_v2.json
V1 elapsed: 476.2 sec; errors: 0
V2 elapsed: 503.72 sec; errors: 0


## 10. Метрики: точное совпадение нормализованного текста + типа

In [10]:
def word_parts(text):
    return re.findall(r"[a-z0-9]+", str(text).lower())

def norm_text(text):
    return " ".join(word_parts(text))

def grouped_from_prediction(pred):
    if pred is None:
        return {t: [] for t in ENTITY_TYPES}
    if "entities" in pred and isinstance(pred["entities"], dict):
        pred = pred["entities"]
    return {t: list(pred.get(t, [])) for t in ENTITY_TYPES}

def ordered_from_prediction(pred):
    if pred is None:
        return []
    entities = pred.get("entities", []) if isinstance(pred, dict) else []
    return [
        {"text": x.get("text", ""), "type": x.get("type", "")}
        for x in entities if isinstance(x, dict)
    ]

def pred_entity_counter(pred, fmt="grouped"):
    counter = Counter()
    if fmt == "grouped":
        grouped = grouped_from_prediction(pred)
        for typ in ENTITY_TYPES:
            for mention in grouped.get(typ, []):
                key = (typ, norm_text(mention))
                if key[1]: counter[key] += 1
    else:
        for item in ordered_from_prediction(pred):
            key = (item["type"], norm_text(item["text"]))
            if key[0] in ENTITY_TYPES and key[1]: counter[key] += 1
    return counter

def gold_entity_counter(global_index):
    return Counter((e["type"], norm_text(e["text"])) for e in gold_entities[global_index] if norm_text(e["text"]))

def score_predictions(predictions, global_positions, fmt="grouped"):
    tp = fp = fn = 0
    evaluated = 0
    for local_i, global_i in enumerate(global_positions):
        evaluated += 1
        pred = predictions[local_i]
        if pred is None:
            fn += sum(gold_entity_counter(global_i).values())
            continue
        p = pred_entity_counter(pred, fmt)
        g = gold_entity_counter(global_i)
        tp += sum((p & g).values())
        fp += sum((p - g).values())
        fn += sum((g - p).values())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": precision, "recall": recall, "f1": f1,
            "tp": tp, "fp": fp, "fn": fn, "evaluated_sentences": evaluated}

def unpack_grouped_result(run_result):
    out = []
    for pred in run_result["predictions"]:
        if pred is None:
            out.append(None)
        elif "entities" in pred:
            out.append(pred["entities"])
        else:
            out.append(pred)
    return out

v1_predictions = unpack_grouped_result(v1_result)
v2_predictions = unpack_grouped_result(v2_result)
metrics_v1 = score_predictions(v1_predictions, test_positions, "grouped")
metrics_v2 = score_predictions(v2_predictions, test_positions, "grouped")

comparison = pd.DataFrame([
    {"experiment": "Qwen / prompt v1 / grouped", **metrics_v1,
     "elapsed_seconds": v1_result["elapsed_seconds"], "errors": len(v1_result["errors"])},
    {"experiment": "Qwen / prompt v2 / grouped", **metrics_v2,
     "elapsed_seconds": v2_result["elapsed_seconds"], "errors": len(v2_result["errors"])},
])
display(comparison)

,experiment,precision,recall,f1,tp,fp,fn,evaluated_sentences,elapsed_seconds,errors
0,Qwen / prompt v1 / grouped,0.357724,0.571429,0.440000,132,237,99,140,476.204906,0
1,Qwen / prompt v2 / grouped,0.494505,0.584416,0.535714,135,138,96,140,503.724166,0


### 10.1. Дополнительная оценка по BIO-разметке через `seqeval`

Entity-level scoring выше сравнивает нормализованный текст и тип. Для проверки границ сущностей дополнительно сопоставляем предсказанные фразы исходным токенам и рассчитываем `seqeval`. Если модель не вернула результат, считаем все токены этого предложения `O`, чтобы ошибка формата не исчезала из оценки. Сопоставление текстовых фраз токенам неоднозначно для повторных/перекрывающихся упоминаний, поэтому интерпретируй этот отчёт вместе с entity-level метриками.

In [ ]:
def mention_records(pred, fmt="grouped"):
    if pred is None:
        return []
    if fmt == "ordered":
        return ordered_from_prediction(pred)
    grouped = grouped_from_prediction(pred)
    return [{"text": text, "type": typ}
            for typ in ENTITY_TYPES for text in grouped.get(typ, [])]

def all_token_spans_for_mention(tokens, mention):
    target = word_parts(mention)
    if not target:
        return []
    flat_parts, flat_to_token = [], []
    for token_i, token in enumerate(tokens):
        parts = word_parts(token)
        flat_parts.extend(parts)
        flat_to_token.extend([token_i] * len(parts))
    spans = []
    for start in range(len(flat_parts) - len(target) + 1):
        if flat_parts[start:start + len(target)] == target:
            spans.append((flat_to_token[start], flat_to_token[start + len(target) - 1] + 1))
    return spans

def predicted_bio_tags(tokens, pred, fmt="grouped"):
    tags = ["O"] * len(tokens)
    occupied = [False] * len(tokens)
    unmatched = []
    records = mention_records(pred, fmt)
    records = sorted(records, key=lambda x: len(word_parts(x.get("text", ""))), reverse=True)
    for record in records:
        text, typ = record.get("text", ""), record.get("type", "")
        if typ not in ENTITY_TYPES:
            unmatched.append({"text": text, "type": typ, "reason": "invalid_type"})
            continue
        candidates = all_token_spans_for_mention(tokens, text)
        span = next((s for s in candidates if not any(occupied[j] for j in range(s[0], s[1]))), None)
        if span is None:
            unmatched.append({"text": text, "type": typ, "reason": "not_found_or_overlap"})
            continue
        start, end = span
        tags[start] = f"B-{typ}"
        for j in range(start + 1, end):
            tags[j] = f"I-{typ}"
        for j in range(start, end):
            occupied[j] = True
    return tags, unmatched

def run_seqeval(predictions, global_positions, fmt="grouped"):
    y_true, y_pred, unmatched_all = [], [], []
    for local_i, global_i in enumerate(global_positions):
        y_true.append(gold_tags[global_i])
        pred = predictions[local_i]
        tags, unmatched = predicted_bio_tags(sentences[global_i], pred, fmt)
        y_pred.append(tags)
        unmatched_all.extend([{"index": global_i, **item} for item in unmatched])
    return {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "report": classification_report(y_true, y_pred, digits=4, zero_division=0),
        "unmatched": unmatched_all,
    }

seqeval_v1 = run_seqeval(v1_predictions, test_positions, "grouped")
seqeval_v2 = run_seqeval(v2_predictions, test_positions, "grouped")
print("Prompt v1 / seqeval: P={:.4f} R={:.4f} F1={:.4f}".format(
    seqeval_v1["precision"], seqeval_v1["recall"], seqeval_v1["f1"]))
print(seqeval_v1["report"])
print("Prompt v2 / seqeval: P={:.4f} R={:.4f} F1={:.4f}".format(
    seqeval_v2["precision"], seqeval_v2["recall"], seqeval_v2["f1"]))
print(seqeval_v2["report"])
print("Unmatched mentions v1/v2:", len(seqeval_v1["unmatched"]), len(seqeval_v2["unmatched"]))

Prompt v1 / seqeval: P=0.3567 R=0.5065 F1=0.4186
              precision    recall  f1-score   support

         LOC     0.4454    0.7465    0.5579        71
        MISC     0.0112    0.0345    0.0169        29
         ORG     0.5333    0.3077    0.3902        78
         PER     0.5200    0.7358    0.6094        53

   micro avg     0.3567    0.5065    0.4186       231
   macro avg     0.3775    0.4561    0.3936       231
weighted avg     0.4377    0.5065    0.4452       231

Prompt v2 / seqeval: P=0.5216 R=0.5238 F1=0.5227
              precision    recall  f1-score   support

         LOC     0.4956    0.7887    0.6087        71
        MISC     0.0000    0.0000    0.0000        29
         ORG     0.5000    0.3077    0.3810        78
         PER     0.7069    0.7736    0.7387        53

   micro avg     0.5216    0.5238    0.5227       231
   macro avg     0.4256    0.4675    0.4321       231
weighted avg     0.4833    0.5238    0.4852       231

Unmatched mentions v1/v2: 41 41


## 11. Отдельная оценка категорий и разбор MISC

In [ ]:
def per_type_metrics(predictions, global_positions, fmt="grouped"):
    rows = []
    for typ in ENTITY_TYPES:
        tp = fp = fn = 0
        for local_i, global_i in enumerate(global_positions):
            pred = predictions[local_i]
            if pred is None:
                fn += sum(1 for e in gold_entities[global_i] if e["type"] == typ)
                continue
            p = Counter()
            if fmt == "grouped":
                d = grouped_from_prediction(pred)
                p.update(norm_text(x) for x in d.get(typ, []) if norm_text(x))
            else:
                p.update(norm_text(x["text"]) for x in ordered_from_prediction(pred)
                         if x["type"] == typ and norm_text(x["text"]))
            g = Counter(norm_text(e["text"]) for e in gold_entities[global_i]
                        if e["type"] == typ and norm_text(e["text"]))
            tp += sum((p & g).values()); fp += sum((p - g).values()); fn += sum((g - p).values())
        pval = tp/(tp+fp) if tp+fp else 0.0
        rval = tp/(tp+fn) if tp+fn else 0.0
        fval = 2*pval*rval/(pval+rval) if pval+rval else 0.0
        rows.append({"type": typ, "precision": pval, "recall": rval, "f1": fval,
                     "tp": tp, "fp": fp, "fn": fn, "gold_support": tp+fn})
    return pd.DataFrame(rows)

print("Prompt v1 by category")
display(per_type_metrics(v1_predictions, test_positions))
print("Prompt v2 by category")
display(per_type_metrics(v2_predictions, test_positions))

misc_examples = []
for global_i in test_positions:
    for e in gold_entities[global_i]:
        if e["type"] == "MISC":
            misc_examples.append({"index": global_i, "sentence": " ".join(sentences[global_i]), "entity": e["text"]})
print("Примеры эталонных MISC (первые 20):")
display(pd.DataFrame(misc_examples).head(20))

Prompt v1 by category


,type,precision,recall,f1,tp,fp,fn,gold_support
0,PER,0.512821,0.754717,0.610687,40,38,13,53
1,LOC,0.426573,0.859155,0.570093,61,82,10,71
2,ORG,0.535714,0.384615,0.447761,30,26,48,78
3,MISC,0.010870,0.034483,0.016529,1,91,28,29


Prompt v2 by category


,type,precision,recall,f1,tp,fp,fn,gold_support
0,PER,0.683333,0.773585,0.725664,41,19,12,53
1,LOC,0.439716,0.873239,0.584906,62,79,9,71
2,ORG,0.542373,0.410256,0.467153,32,27,46,78
3,MISC,0.000000,0.000000,0.000000,0,13,29,29


Примеры эталонных MISC (первые 20):


,index,sentence,entity
0,19,"The television , which did not say when the se...",French
1,19,"The television , which did not say when the se...",Roman Catholic
2,165,The former Soviet Union was displaced as contr...,International Petroleum Encyclopedia
3,104,U.S. Open officials managed to insult most of ...,U.S. Open
4,63,British hostage in Chechnya describes ordeal .,British
5,100,The tiny islands of the Seychelles failed to m...,African Nations Cup
6,27,"New York Dow Jones industrial average -- 5,778...",Dow Jones
7,184,BASEBALL - MAJOR LEAGUE STANDINGS AFTER WEDNES...,MAJOR LEAGUE
8,112,They said soldiers stole a gold necklace and a...,Israeli
9,174,Gold miner Newcrest Mining Ltd said on Wednesd...,A$


## 12. Качественный анализ: пропуски, ложные сущности и неверные типы

In [13]:
def error_analysis(predictions, global_positions, fmt="grouped", limit=40):
    rows = []
    for local_i, global_i in enumerate(global_positions):
        pred = predictions[local_i]
        gold = gold_entity_counter(global_i)
        if pred is None:
            rows.append({"index": global_i, "sentence": " ".join(sentences[global_i]),
                         "error_type": "INVALID/FAILED OUTPUT", "gold": list(gold.elements()), "prediction": None})
            continue
        p = pred_entity_counter(pred, fmt)
        missed = list((gold - p).elements())
        extra = list((p - gold).elements())
        if missed or extra:
            rows.append({"index": global_i, "sentence": " ".join(sentences[global_i]),
                         "error_type": "MISS + FALSE POSITIVE" if missed and extra else ("MISS" if missed else "FALSE POSITIVE"),
                         "missed": missed, "extra_or_wrong_type": extra,
                         "gold_entities": [ (e["text"], e["type"]) for e in gold_entities[global_i] ],
                         "predicted_entities": list(p.elements())})
    return pd.DataFrame(rows).head(limit)

print("Ошибки prompt v2:")
display(error_analysis(v2_predictions, test_positions, "grouped"))

print("Отдельно ошибки, связанные с MISC:")
misc_error_rows = []
for row in error_analysis(v2_predictions, test_positions, "grouped", limit=1000).to_dict("records"):
    if any(item[0] == "MISC" for item in row.get("missed", [])) or any(item[0] == "MISC" for item in row.get("extra_or_wrong_type", [])):
        misc_error_rows.append(row)
display(pd.DataFrame(misc_error_rows).head(30))

Ошибки prompt v2:


,index,sentence,error_type,missed,extra_or_wrong_type,gold_entities,predicted_entities
0,102,Sporting Sale 0 Meknes 1,MISS + FALSE POSITIVE,"[(ORG, meknes)]","[(LOC, meknes)]","[(Sporting Sale, ORG), (Meknes, ORG)]","[(LOC, meknes), (ORG, sporting sale)]"
1,38,THU M&R YR DIV 31.7 10.5-42.3 47.0,MISS,"[(ORG, m r)]",[],"[(M&R, ORG)]",[]
2,156,"Chile 's ENAP buys Oriente , Escravos crude fo...",MISS + FALSE POSITIVE,"[(LOC, chile), (ORG, oriente), (ORG, escravos)]","[(PER, chile), (LOC, enap), (LOC, oriente), (L...","[(Chile, LOC), (ENAP, ORG), (Oriente, ORG), (E...","[(PER, chile), (LOC, enap), (LOC, oriente), (L..."
3,135,were largely unchanged at the Bank of Finland ...,FALSE POSITIVE,[],"[(LOC, bank of finland)]","[(Bank of Finland, ORG)]","[(LOC, bank of finland), (ORG, bank of finland)]"
4,19,"The television , which did not say when the se...",MISS + FALSE POSITIVE,"[(MISC, french), (MISC, roman catholic)]","[(LOC, france), (ORG, television)]","[(French, MISC), (Roman Catholic, MISC), (Pier...","[(PER, pierre claverie), (LOC, france), (ORG, ..."
5,177,"-- Steve Weizman , Copenhagen newsroom +45 339...",FALSE POSITIVE,[],"[(ORG, newsroom)]","[(Steve Weizman, PER), (Copenhagen, LOC)]","[(PER, steve weizman), (LOC, copenhagen), (ORG..."
6,146,"Women 's singles , second round",FALSE POSITIVE,[],"[(MISC, women s singles)]",[],"[(MISC, women s singles)]"
7,179,5. Giunluca Gorini -LPR- Italy -RPR- Aki 45,MISS,"[(ORG, aki)]",[],"[(Giunluca Gorini, PER), (Italy, LOC), (Aki, O...","[(PER, giunluca gorini), (LOC, italy)]"
8,65,QADISSIYA,MISS + FALSE POSITIVE,"[(ORG, qadissiya)]","[(PER, qadissiya)]","[(QADISSIYA, ORG)]","[(PER, qadissiya)]"
9,84,Zurich 9 2 5 2 9 9 11,MISS + FALSE POSITIVE,"[(ORG, zurich)]","[(LOC, zurich)]","[(Zurich, ORG)]","[(LOC, zurich)]"


Отдельно ошибки, связанные с MISC:


,index,sentence,error_type,missed,extra_or_wrong_type,gold_entities,predicted_entities
0,19,"The television , which did not say when the se...",MISS + FALSE POSITIVE,"[(MISC, french), (MISC, roman catholic)]","[(LOC, france), (ORG, television)]","[(French, MISC), (Roman Catholic, MISC), (Pier...","[(PER, pierre claverie), (LOC, france), (ORG, ..."
1,146,"Women 's singles , second round",FALSE POSITIVE,[],"[(MISC, women s singles)]",[],"[(MISC, women s singles)]"
2,165,The former Soviet Union was displaced as contr...,MISS + FALSE POSITIVE,"[(LOC, shabwa), (MISC, international petroleum...","[(LOC, shabwa oilfields), (LOC, international ...","[(Soviet Union, LOC), (Shabwa, LOC), (Internat...","[(LOC, soviet union), (LOC, shabwa oilfields),..."
3,104,U.S. Open officials managed to insult most of ...,MISS + FALSE POSITIVE,"[(MISC, u s open)]","[(LOC, u s), (ORG, u s open)]","[(U.S. Open, MISC)]","[(LOC, u s), (ORG, u s open)]"
4,63,British hostage in Chechnya describes ordeal .,MISS + FALSE POSITIVE,"[(MISC, british)]","[(PER, british)]","[(British, MISC), (Chechnya, LOC)]","[(PER, british), (LOC, chechnya)]"
5,100,The tiny islands of the Seychelles failed to m...,MISS + FALSE POSITIVE,"[(MISC, african nations cup)]","[(LOC, african nations cup)]","[(Seychelles, LOC), (African Nations Cup, MISC)]","[(LOC, seychelles), (LOC, african nations cup)]"
6,27,"New York Dow Jones industrial average -- 5,778...",MISS + FALSE POSITIVE,"[(MISC, dow jones)]","[(ORG, dow jones industrial average)]","[(New York, LOC), (Dow Jones, MISC)]","[(LOC, new york), (ORG, dow jones industrial a..."
7,184,BASEBALL - MAJOR LEAGUE STANDINGS AFTER WEDNES...,MISS + FALSE POSITIVE,"[(MISC, major league)]","[(ORG, major league standings)]","[(MAJOR LEAGUE, MISC)]","[(ORG, major league standings)]"
8,112,They said soldiers stole a gold necklace and a...,MISS + FALSE POSITIVE,"[(MISC, israeli)]","[(PER, soldiers), (PER, an elderly woman), (PE...","[(Israeli, MISC)]","[(PER, soldiers), (PER, an elderly woman), (PE..."
9,174,Gold miner Newcrest Mining Ltd said on Wednesd...,MISS,"[(MISC, a)]",[],"[(Newcrest Mining Ltd, ORG), (A$, MISC)]","[(ORG, newcrest mining ltd)]"


## 13. DSPy: автоматическая оптимизация инструкции через MIPROv2

In [ ]:
dspy_model_name = f"openai/{PRIMARY_MODEL_ID.lstrip('/')}"
task_lm = dspy.LM(
    dspy_model_name,
    api_base=PRIMARY_BASE_URL,
    api_key=API_KEY,
    model_type="chat",
    temperature=0.0,
    max_tokens=512,
)
prompt_lm = dspy.LM(
    dspy_model_name,
    api_base=PRIMARY_BASE_URL,
    api_key=API_KEY,
    model_type="chat",
    temperature=0.7,
    max_tokens=512,
)
dspy.configure(lm=task_lm)
print("DSPy task LM:", dspy_model_name)

DSPy task LM: openai/Users/veronika_steklo/Models/qwen2.5-7b-instruct-q3_k_m.gguf


In [15]:
class NERSignature(dspy.Signature):
    """Extract all named entities from one sentence using CoNLL-2003 types.

    Return one valid JSON object with keys PER, LOC, ORG, MISC; each value is a list
    of exact entity strings. PER=person names, LOC=geographical places, ORG=organizations,
    MISC=nationalities/ethnic or political groups/languages/events and other named entities
    outside the first three categories. Exclude dates, years, scores, ordinary nouns,
    occupations, titles, punctuation, and dataset markup. Never invent, expand, or duplicate
    entities. Copy exact spans from the sentence. Empty categories must be empty lists.
    """
    sentence: str = dspy.InputField(desc="Original sentence from CoNLL-2003")
    entities_json: str = dspy.OutputField(
        desc='Valid JSON object: {"PER": [], "LOC": [], "ORG": [], "MISC": []}'
    )

class NERProgram(dspy.Module):
    def __init__(self):
        super().__init__()
        self.extract = dspy.Predict(NERSignature)
    def forward(self, sentence):
        return self.extract(sentence=sentence)

ner_program = NERProgram()

def parse_grouped_json(raw):
    try:
        data = extract_json_from_text(raw)
        if "entities" in data and isinstance(data["entities"], dict):
            data = data["entities"]
        if not isinstance(data, dict):
            return None
        return {t: [x for x in data.get(t, []) if isinstance(x, str)]
                if isinstance(data.get(t, []), list) else [] for t in ENTITY_TYPES}
    except Exception:
        return None

def dspy_metric(example, prediction, trace=None):
    pred = parse_grouped_json(getattr(prediction, "entities_json", ""))
    if pred is None:
        return 0.0
    gold = json.loads(example.gold_json)
    p = Counter((t, norm_text(x)) for t in ENTITY_TYPES for x in pred[t] if norm_text(x))
    g = Counter((t, norm_text(x)) for t in ENTITY_TYPES for x in gold[t] if norm_text(x))
    tp = sum((p & g).values()); fp = sum((p - g).values()); fn = sum((g - p).values())
    precision = tp/(tp+fp) if tp+fp else 0.0
    recall = tp/(tp+fn) if tp+fn else 0.0
    return 2*precision*recall/(precision+recall) if precision+recall else 0.0

def dspy_example(global_i):
    return dspy.Example(
        sentence=" ".join(sentences[global_i]),
        gold_json=json.dumps(group_gold(gold_entities[global_i]), ensure_ascii=False),
    ).with_inputs("sentence")

trainset = [dspy_example(i) for i in train_positions]
devset = [dspy_example(i) for i in dev_positions]
print("DSPy train/dev:", len(trainset), len(devset))

DSPy train/dev: 20 40


In [16]:
optimized_program = None
optimized_cache = OUTPUT_DIR / f"{MODEL_SLUG}_dspy_ner_grouped_optimized.json"
if RUN_DSPY_OPTIMIZATION:
    if optimized_cache.exists():
        print("Загружаем сохранённую оптимизацию:", optimized_cache)
        optimized_program = NERProgram()
        optimized_program.load(str(optimized_cache))
    else:
        optimizer = dspy.MIPROv2(
            metric=dspy_metric,
            prompt_model=prompt_lm,
            auto=DSPY_AUTO_MODE,
            num_threads=1,
            verbose=True,
        )
        optimized_program = optimizer.compile(
            NERProgram(),
            trainset=trainset,
            valset=devset,
            max_bootstrapped_demos=0,
            max_labeled_demos=0,
        )
        optimized_program.save(str(optimized_cache))
        print("Сохранено:", optimized_cache)
    print("Оптимизированные инструкции:")
    print(optimized_program.extract.signature.instructions)
else:
    print("RUN_DSPY_OPTIMIZATION=False: пропускаем DSPy compile.")

2026/10/10 16:13:50 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 9
minibatch: False
num_fewshot_candidates: 6
num_instruct_candidates: 6
valset size: 40

2026/10/10 16:13:50 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2026/10/10 16:13:50 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used for informing instruction proposal.

2026/10/10 16:13:50 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=6 sets of demonstrations...


Bootstrapping set 1/6
Bootstrapping set 2/6


 10%|█         | 2/20 [00:00<00:01, 13.92it/s]


Bootstrapped 1 full traces after 2 examples for up to 1 rounds, amounting to 2 attempts.
Bootstrapping set 3/6


 40%|████      | 8/20 [00:00<00:00, 73.89it/s]


...


Predictor 0
i: In a high-stakes investigation, you must extract all named entities from the following sentence using CoNLL-2003 types. The accuracy of your JSON output can mean the difference between solving a major case and failing to identify key individuals and locations. 

Return one valid JSON object with keys PER, LOC, ORG, MISC; each value is a list
of exact entity strings. PER=person names, LOC=geographical places, ORG=organizations,
MISC=nationalities/ethnic or political groups/languages/events and other named entities
outside the first three categories. Exclude dates, years, scores, ordinary nouns,
occupations, titles, punctuation, and dataset markup. Never invent, expand, or duplicate
entities. Copy exact spans from the sentence. Empty categories must be empty lists.

{
  "PER": [],
  "LOC": [],
  "ORG": [],
  "MISC": []
}
p: Entities Json:


Average Metric: 16.25 / 40 (40.6%): 100%|██████████| 40/40 [00:00<00:00, 47.13it/s]

2026/10/10 16:27:11 INFO dspy.evaluate.evaluate: Average Metric: 16.2492784992785 / 40 (40.6%)
2026/10/10 16:27:11 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 40.62 with parameters ['Predictor 0: Instruction 2'].
2026/10/10 16:27:11 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [33.72, 36.48, 36.03, 33.72, 37.93, 40.62, 40.62, 33.72, 40.62]
2026/10/10 16:27:11 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 40.62
2026/10/10 16:27:11 INFO dspy.teleprompt.mipro_optimizer_v2: =======================


2026/10/10 16:27:11 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 10 / 9 =====
2026/10/10 16:27:11 INFO dspy.teleprompt.mipro_optimizer_v2: Evaluating the following candidate program...




Predictor 0
i: Extract all named entities from one sentence using CoNLL-2003 types.

Return one valid JSON object with keys PER, LOC, ORG, MISC; each value is a list
of exact entity strings. PER=person names, LOC=geographical places, ORG=organizations,
MISC=nationalities/ethnic or political groups/languages/events and other named entities
outside the first three categories. Exclude dates, years, scores, ordinary nouns,
occupations, titles, punctuation, and dataset markup. Never invent, expand, or duplicate
entities. Copy exact spans from the sentence. Empty categories must be empty lists.
p: Entities Json:


Average Metric: 13.49 / 40 (33.7%): 100%|██████████| 40/40 [00:00<00:00, 52.52it/s]

2026/10/10 16:27:12 INFO dspy.evaluate.evaluate: Average Metric: 13.486385836385836 / 40 (33.7%)
2026/10/10 16:27:12 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 33.72 with parameters ['Predictor 0: Instruction 0'].
2026/10/10 16:27:12 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [33.72, 36.48, 36.03, 33.72, 37.93, 40.62, 40.62, 33.72, 40.62, 33.72]
2026/10/10 16:27:12 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 40.62
2026/10/10 16:27:12 INFO dspy.teleprompt.mipro_optimizer_v2: ========================


2026/10/10 16:27:12 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 40.62!



Сохранено: ner_outputs/Users_veronika_steklo_Models_qwen2.5-7b-instruct-q3_k_m.gguf_dspy_ner_grouped_optimized.json
Оптимизированные инструкции:
In a high-stakes investigation, you must extract all named entities from the following sentence using CoNLL-2003 types. The accuracy of your JSON output can mean the difference between solving a major case and failing to identify key individuals and locations. 

Return one valid JSON object with keys PER, LOC, ORG, MISC; each value is a list
of exact entity strings. PER=person names, LOC=geographical places, ORG=organizations,
MISC=nationalities/ethnic or political groups/languages/events and other named entities
outside the first three categories. Exclude dates, years, scores, ordinary nouns,
occupations, titles, punctuation, and dataset markup. Never invent, expand, or duplicate
entities. Copy exact spans from the sentence. Empty categories must be empty lists.

{
  "PER": [],
  "LOC": [],
  "ORG": [],
  "MISC": []
}


## 14. Оцениваем DSPy-оптимизированный промпт на том же test

In [17]:
dspy_group_predictions = [None] * len(test_positions)
dspy_group_raw = [None] * len(test_positions)
dspy_group_errors = []
dspy_group_elapsed = 0.0

if optimized_program is not None:
    start = time.perf_counter()
    for local_i, global_i in enumerate(test_positions):
        try:
            result = optimized_program(sentence=" ".join(sentences[global_i]))
            raw = result.entities_json
            parsed = parse_grouped_json(raw)
            dspy_group_raw[local_i] = raw
            dspy_group_predictions[local_i] = parsed
            if parsed is None:
                dspy_group_errors.append({"index": global_i, "raw": raw, "error": "Invalid JSON"})
        except Exception as exc:
            dspy_group_errors.append({"index": global_i, "error": repr(exc)})
    dspy_group_elapsed = time.perf_counter() - start
    (OUTPUT_DIR / "dspy_grouped_test_raw.json").write_text(
        json.dumps({"raw": dspy_group_raw, "errors": dspy_group_errors}, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )
    print("DSPy test seconds:", round(dspy_group_elapsed, 2), "errors:", len(dspy_group_errors))
else:
    print("DSPy optimization not run; no DSPy test predictions.")

rows = [
    {"experiment": "Qwen prompt v1 / grouped", **metrics_v1,
     "elapsed_seconds": v1_result["elapsed_seconds"], "invalid_outputs": len(v1_result["errors"])},
    {"experiment": "Qwen prompt v2 / grouped", **metrics_v2,
     "elapsed_seconds": v2_result["elapsed_seconds"], "invalid_outputs": len(v2_result["errors"])},
]
if optimized_program is not None:
    dspy_group_metrics = score_predictions(dspy_group_predictions, test_positions, "grouped")
    rows.append({"experiment": "Qwen DSPy MIPROv2 / grouped", **dspy_group_metrics,
                 "elapsed_seconds": dspy_group_elapsed, "invalid_outputs": len(dspy_group_errors)})
display(pd.DataFrame(rows))

DSPy test seconds: 551.8 errors: 3


,experiment,precision,recall,f1,tp,fp,fn,evaluated_sentences,elapsed_seconds,invalid_outputs
0,Qwen prompt v1 / grouped,0.357724,0.571429,0.440000,132,237,99,140,476.204906,0
1,Qwen prompt v2 / grouped,0.494505,0.584416,0.535714,135,138,96,140,503.724166,0
2,Qwen DSPy MIPROv2 / grouped,0.578704,0.541126,0.559284,125,91,106,140,551.803688,3


## 15. Новый формат Structured Output: единый список по порядку появления

In [18]:
ORDERED_PROMPT = PROMPT_V2 + """
Output format: return a JSON object with one key, \"entities\". Its value is a list of
objects with exactly the fields \"text\" and \"type\". Sort this list by each entity's
first character position in the original sentence, from left to right. Preserve repeated
mentions as separate list items. Do not group by type. If there are no entities, return
{\"entities\": []}.
"""

cache_ordered = CACHE_DIR / f"{MODEL_SLUG}_ordered_prompt_v2.json"
ordered_result = await load_or_run(cache_ordered, primary_async_client, PRIMARY_MODEL_ID,
                                   test_tokens, ORDERED_PROMPT, OrderedNERResult)
ordered_predictions = ordered_result["predictions"]
ordered_metrics = score_predictions(ordered_predictions, test_positions, "ordered")
print("Ordered format metrics:")
print(ordered_metrics)
print("Elapsed:", round(ordered_result["elapsed_seconds"], 2), "sec; errors:", len(ordered_result["errors"]))

Запускаю модель; это может занять несколько минут: Users_veronika_steklo_Models_qwen2.5-7b-instruct-q3_k_m.gguf_ordered_prompt_v2.json
Ordered format metrics:
{'precision': 0.5545454545454546, 'recall': 0.5281385281385281, 'f1': 0.541019955654102, 'tp': 122, 'fp': 98, 'fn': 109, 'evaluated_sentences': 140}
Elapsed: 421.61 sec; errors: 0


## 16. Проверяем порядок сущностей

In [ ]:
def order_metrics(ordered_predictions, global_positions):
    rows = []
    exact_sequence = 0
    total = 0
    pair_correct = pair_total = 0
    for local_i, global_i in enumerate(global_positions):
        pred = ordered_predictions[local_i]
        predicted = [(x["type"], norm_text(x["text"])) for x in ordered_from_prediction(pred)
                     if x["type"] in ENTITY_TYPES and norm_text(x["text"])]
        gold = [(e["type"], norm_text(e["text"])) for e in sorted(gold_entities[global_i], key=lambda x: x["start"])
                if norm_text(e["text"])]
        total += 1
        is_exact = (pred is not None) and (predicted == gold)
        exact_sequence += int(is_exact)
        common = Counter(predicted) & Counter(gold)
        matched = []
        used = Counter()
        for pidx, item in enumerate(predicted):
            if used[item] < common[item]:
                gidx = next((j for j, gitem in enumerate(gold) if gitem == item and
                             sum(1 for prior in gold[:j] if prior == item) == used[item]), None)
                if gidx is not None:
                    matched.append((pidx, gidx))
                    used[item] += 1
        for a in range(len(matched)):
            for b in range(a + 1, len(matched)):
                pair_total += 1
                pair_correct += int((matched[a][0] < matched[b][0]) == (matched[a][1] < matched[b][1]))
        rows.append({"index": global_i, "exact_order_and_entities": is_exact,
                     "predicted_count": len(predicted), "gold_count": len(gold),
                     "predicted_sequence": predicted, "gold_sequence": gold})
    return {
        "sentences_evaluated": total,
        "exact_sequence_accuracy": exact_sequence / total if total else 0.0,
        "pairwise_order_accuracy": pair_correct / pair_total if pair_total else None,
        "pairwise_comparisons": pair_total,
        "details": pd.DataFrame(rows),
    }

ordered_order = order_metrics(ordered_predictions, test_positions)
print("Exact entity sequence accuracy:", round(ordered_order["exact_sequence_accuracy"], 4))
print("Pairwise order accuracy among matched entities:", ordered_order["pairwise_order_accuracy"])
display(ordered_order["details"].head(20))

Exact entity sequence accuracy: 0.3643
Pairwise order accuracy among matched entities: 0.9583333333333334


,index,exact_order_and_entities,predicted_count,gold_count,predicted_sequence,gold_sequence
0,102,False,2,2,"[(LOC, meknes), (ORG, sporting sale)]","[(ORG, sporting sale), (ORG, meknes)]"
1,38,False,0,1,[],"[(ORG, m r)]"
2,156,False,4,4,"[(LOC, chile), (ORG, enap), (MISC, oriente), (...","[(LOC, chile), (ORG, enap), (ORG, oriente), (O..."
3,176,False,2,1,"[(LOC, iceland), (LOC, each alliance country)]","[(LOC, iceland)]"
4,135,True,1,1,"[(ORG, bank of finland)]","[(ORG, bank of finland)]"
5,78,False,1,3,"[(PER, yuuchi takeda)]","[(PER, yuuchi takeda), (LOC, japan), (ORG, hon..."
6,115,True,0,0,[],[]
7,19,False,4,3,"[(ORG, television), (ORG, security forces), (P...","[(MISC, french), (MISC, roman catholic), (PER,..."
8,14,True,2,2,"[(PER, daniel vacek), (LOC, czech republic)]","[(PER, daniel vacek), (LOC, czech republic)]"
9,177,True,2,2,"[(PER, steve weizman), (LOC, copenhagen)]","[(PER, steve weizman), (LOC, copenhagen)]"


### 16.1. Качественная оценка ordered-формата

Помимо порядка, повторяем оценку по категориям и смотрим ошибки именно для списка сущностей. Здесь особенно важно заметить, если модель нашла сущность, но поставила её не на то место или выдала её в неверном типе.

In [20]:
print("Ordered format by entity type:")
display(per_type_metrics(ordered_predictions, test_positions, "ordered"))
print("Ordered format errors:")
display(error_analysis(ordered_predictions, test_positions, "ordered", limit=40))
seqeval_ordered = run_seqeval(ordered_predictions, test_positions, "ordered")
print("Ordered / seqeval: P={:.4f} R={:.4f} F1={:.4f}".format(
    seqeval_ordered["precision"], seqeval_ordered["recall"], seqeval_ordered["f1"]))
print(seqeval_ordered["report"])

Ordered format by entity type:


,type,precision,recall,f1,tp,fp,fn,gold_support
0,PER,0.733333,0.622642,0.673469,33,12,20,53
1,LOC,0.627907,0.760563,0.687898,54,32,17,71
2,ORG,0.607143,0.435897,0.507463,34,22,44,78
3,MISC,0.030303,0.034483,0.032258,1,32,28,29


Ordered format errors:


,index,sentence,error_type,missed,extra_or_wrong_type,gold_entities,predicted_entities
0,102,Sporting Sale 0 Meknes 1,MISS + FALSE POSITIVE,"[(ORG, meknes)]","[(LOC, meknes)]","[(Sporting Sale, ORG), (Meknes, ORG)]","[(LOC, meknes), (ORG, sporting sale)]"
1,38,THU M&R YR DIV 31.7 10.5-42.3 47.0,MISS,"[(ORG, m r)]",[],"[(M&R, ORG)]",[]
2,156,"Chile 's ENAP buys Oriente , Escravos crude fo...",MISS + FALSE POSITIVE,"[(ORG, oriente), (ORG, escravos)]","[(MISC, oriente), (MISC, escravos)]","[(Chile, LOC), (ENAP, ORG), (Oriente, ORG), (E...","[(LOC, chile), (ORG, enap), (MISC, oriente), (..."
3,176,The committee consists of the chiefs of defenc...,FALSE POSITIVE,[],"[(LOC, each alliance country)]","[(Iceland, LOC)]","[(LOC, iceland), (LOC, each alliance country)]"
4,78,1. Yuuchi Takeda -LPR- Japan -RPR- Honda 38 mi...,MISS,"[(LOC, japan), (ORG, honda)]",[],"[(Yuuchi Takeda, PER), (Japan, LOC), (Honda, O...","[(PER, yuuchi takeda)]"
5,19,"The television , which did not say when the se...",MISS + FALSE POSITIVE,"[(MISC, french), (MISC, roman catholic), (PER,...","[(ORG, television), (ORG, security forces), (P...","[(French, MISC), (Roman Catholic, MISC), (Pier...","[(ORG, television), (ORG, security forces), (P..."
6,146,"Women 's singles , second round",FALSE POSITIVE,[],"[(MISC, women s singles), (MISC, second round)]",[],"[(MISC, women s singles), (MISC, second round)]"
7,179,5. Giunluca Gorini -LPR- Italy -RPR- Aki 45,MISS,"[(LOC, italy), (ORG, aki)]",[],"[(Giunluca Gorini, PER), (Italy, LOC), (Aki, O...","[(PER, giunluca gorini)]"
8,65,QADISSIYA,MISS + FALSE POSITIVE,"[(ORG, qadissiya)]","[(MISC, qadissiya)]","[(QADISSIYA, ORG)]","[(MISC, qadissiya)]"
9,84,Zurich 9 2 5 2 9 9 11,MISS + FALSE POSITIVE,"[(ORG, zurich)]","[(LOC, zurich)]","[(Zurich, ORG)]","[(LOC, zurich)]"


Ordered / seqeval: P=0.5550 R=0.5238 F1=0.5390
              precision    recall  f1-score   support

         LOC     0.6310    0.7465    0.6839        71
        MISC     0.0303    0.0345    0.0323        29
         ORG     0.6071    0.4359    0.5075        78
         PER     0.7333    0.6226    0.6735        53

   micro avg     0.5550    0.5238    0.5390       231
   macro avg     0.5004    0.4599    0.4743       231
weighted avg     0.5710    0.5238    0.5401       231



## 17. Дополнительная DSPy-оптимизация для ordered-формата (необязательная, но готовая)

In [21]:
class OrderedNERSignature(dspy.Signature):
    """Extract all named entities using CoNLL-2003 types and return them in left-to-right order.

    Output valid JSON with key entities, whose value is a list of objects {text, type}.
    Preserve exact spans and original order. PER=person names, LOC=geographical places,
    ORG=organizations, MISC=nationalities/ethnic or political groups/languages/events and
    other named entities. Exclude dates, years, scores, generic nouns, occupations, titles,
    punctuation, and dataset markup. Do not invent, merge, expand, or duplicate mentions.
    """
    sentence: str = dspy.InputField()
    entities_json: str = dspy.OutputField(
        desc='JSON object: {"entities": [{"text": "London", "type": "LOC"}]}; sorted left to right'
    )

class OrderedNERProgram(dspy.Module):
    def __init__(self):
        super().__init__()
        self.extract = dspy.Predict(OrderedNERSignature)
    def forward(self, sentence):
        return self.extract(sentence=sentence)

def dspy_ordered_metric(example, prediction, trace=None):
    try:
        data = extract_json_from_text(prediction.entities_json)
        pred = data.get("entities", [])
        gold = json.loads(example.gold_ordered_json)
        p = Counter((x.get("type"), norm_text(x.get("text", ""))) for x in pred if isinstance(x, dict))
        g = Counter((x.get("type"), norm_text(x.get("text", ""))) for x in gold)
        tp = sum((p & g).values()); fp = sum((p - g).values()); fn = sum((g - p).values())
        precision = tp/(tp+fp) if tp+fp else 0.0
        recall = tp/(tp+fn) if tp+fn else 0.0
        return 2*precision*recall/(precision+recall) if precision+recall else 0.0
    except Exception:
        return 0.0

ordered_trainset = [dspy.Example(
    sentence=" ".join(sentences[i]),
    gold_ordered_json=json.dumps([
        {"text": e["text"], "type": e["type"]}
        for e in sorted(gold_entities[i], key=lambda x: x["start"])
    ], ensure_ascii=False)
).with_inputs("sentence") for i in train_positions]
ordered_devset = [dspy.Example(
    sentence=" ".join(sentences[i]),
    gold_ordered_json=json.dumps([
        {"text": e["text"], "type": e["type"]}
        for e in sorted(gold_entities[i], key=lambda x: x["start"])
    ], ensure_ascii=False)
).with_inputs("sentence") for i in dev_positions]

optimized_ordered_program = None
if RUN_ORDERED_DSPY_OPTIMIZATION:
    ordered_optimizer = dspy.MIPROv2(
        metric=dspy_ordered_metric, prompt_model=prompt_lm,
        auto=DSPY_AUTO_MODE, num_threads=1, verbose=True
    )
    optimized_ordered_program = ordered_optimizer.compile(
        OrderedNERProgram(), trainset=ordered_trainset, valset=ordered_devset,
        max_bootstrapped_demos=0, max_labeled_demos=0
    )
    ordered_dspy_path = OUTPUT_DIR / f"{MODEL_SLUG}_dspy_ner_ordered_optimized.json"
    optimized_ordered_program.save(str(ordered_dspy_path))
    print("Saved:", ordered_dspy_path)
    print(optimized_ordered_program.extract.signature.instructions)
    optimized_ordered_predictions = [None] * len(test_positions)
    optimized_ordered_errors = []
    start_ordered_dspy = time.perf_counter()
    for local_i, global_i in enumerate(test_positions):
        try:
            raw = optimized_ordered_program(sentence=" ".join(sentences[global_i])).entities_json
            data = extract_json_from_text(raw)
            optimized_ordered_predictions[local_i] = OrderedNERResult.model_validate(data).model_dump()
        except Exception as exc:
            optimized_ordered_errors.append({"index": global_i, "error": repr(exc)})
    optimized_ordered_elapsed = time.perf_counter() - start_ordered_dspy
    optimized_ordered_metrics = score_predictions(optimized_ordered_predictions, test_positions, "ordered")
    optimized_ordered_order = order_metrics(optimized_ordered_predictions, test_positions)
    print("DSPy ordered metrics:", optimized_ordered_metrics)
    print("DSPy ordered exact sequence accuracy:", optimized_ordered_order["exact_sequence_accuracy"])
    print("DSPy ordered elapsed:", round(optimized_ordered_elapsed, 2), "sec; errors:", len(optimized_ordered_errors))
else:
    optimized_ordered_predictions = None
    optimized_ordered_metrics = None
    optimized_ordered_order = None
    optimized_ordered_errors = []
    print("Пропускаем дополнительную оптимизацию ordered-формата. Для запуска установи RUN_ORDERED_DSPY_OPTIMIZATION=True.")

Пропускаем дополнительную оптимизацию ordered-формата. Для запуска установи RUN_ORDERED_DSPY_OPTIMIZATION=True.


## 18. Сравнение двух форматов результата

In [22]:
format_comparison = pd.DataFrame([
    {"format": "Grouped by type", "schema": '{"entities":{"PER":[],"LOC":[],"ORG":[],"MISC":[]}}',
     **metrics_v2, "elapsed_seconds": v2_result["elapsed_seconds"], "invalid_outputs": len(v2_result["errors"]),
     "order_available": False},
    {"format": "Ordered list", "schema": '{"entities":[{"text":"...","type":"LOC"}]}',
     **ordered_metrics, "elapsed_seconds": ordered_result["elapsed_seconds"], "invalid_outputs": len(ordered_result["errors"]),
     "order_available": True},
])
display(format_comparison)

,format,schema,precision,recall,f1,tp,fp,fn,evaluated_sentences,elapsed_seconds,invalid_outputs,order_available
0,Grouped by type,"{""entities"":{""PER"":[],""LOC"":[],""ORG"":[],""MISC""...",0.494505,0.584416,0.535714,135,138,96,140,503.724166,0,False
1,Ordered list,"{""entities"":[{""text"":""..."",""type"":""LOC""}]}",0.554545,0.528139,0.541020,122,98,109,140,421.610046,0,True


### Интерпретация форматов

- **Группировка по типам** удобна, если следующему этапу системы нужны коллекции людей, мест, организаций и MISC. Но формат теряет исходный порядок и может схлопывать повторные упоминания; труднее восстановить границы и расположение сущности в тексте.
- **Единый упорядоченный список** сохраняет последовательность упоминаний и естественно поддерживает повторные упоминания. Его проще расширить полями `start`/`end` (offsets), но модель может ошибаться в порядке, а формат требует явной проверки `type` и валидности каждого элемента.
- Для production NER предпочтительно сохранять offsets (`start`, `end`) вместе с типом: тогда не нужно повторно искать упоминание в строке. В этом эксперименте сравнение идёт по тексту сущности и типу, потому что модель возвращает текстовые фрагменты.

## 19. Вторая локальная модель: сравнение на тех же тестовых примерах

In [23]:
second_result_grouped = None
second_result_ordered = None
second_metrics_grouped = None
second_metrics_ordered = None
second_model_id_detected = None

if RUN_SECOND_MODEL:
    try:
        second_client = OpenAI(base_url=SECOND_BASE_URL, api_key=API_KEY, timeout=120.0)
        second_async_client = AsyncOpenAI(base_url=SECOND_BASE_URL, api_key=API_KEY, timeout=180.0)
        second_models = second_client.models.list()
        assert second_models.data, f"Нет моделей на {SECOND_BASE_URL}"
        second_model_id_detected = SECOND_MODEL_ID or second_models.data[0].id
        print("Second model:", second_model_id_detected)

        second_slug = re.sub(r"[^A-Za-z0-9._-]+", "_", second_model_id_detected).strip("_") or "second_model"
        second_cache_grouped = CACHE_DIR / f"{second_slug}_grouped_prompt_v2.json"
        second_cache_ordered = CACHE_DIR / f"{second_slug}_ordered_prompt_v2.json"
        second_result_grouped = await load_or_run(
            second_cache_grouped, second_async_client, second_model_id_detected,
            test_tokens, PROMPT_V2, GroupedNERResult
        )
        second_result_ordered = await load_or_run(
            second_cache_ordered, second_async_client, second_model_id_detected,
            test_tokens, ORDERED_PROMPT, OrderedNERResult
        )
        second_predictions_grouped = unpack_grouped_result(second_result_grouped)
        second_predictions_ordered = second_result_ordered["predictions"]
        second_metrics_grouped = score_predictions(second_predictions_grouped, test_positions, "grouped")
        second_metrics_ordered = score_predictions(second_predictions_ordered, test_positions, "ordered")
    except Exception as exc:
        print("Не удалось запустить вторую модель. Проверь endpoint/порт и настройку сервера.")
        print("Ошибка:", repr(exc))
else:
    print("Вторая модель отключена. Запусти её локально на SECOND_BASE_URL, затем поставь RUN_SECOND_MODEL=True и перезапусти эту ячейку.")

Second model: /Users/veronika_steklo/Models/gemma-4-E2B_q4_0-it.gguf
Запускаю модель; это может занять несколько минут: Users_veronika_steklo_Models_gemma-4-E2B_q4_0-it.gguf_grouped_prompt_v2.json
Запускаю модель; это может занять несколько минут: Users_veronika_steklo_Models_gemma-4-E2B_q4_0-it.gguf_ordered_prompt_v2.json


## 20. Итоговая таблица моделей и затрат времени

In [24]:
final_rows = [
    {"model": PRIMARY_MODEL_ID, "experiment": "Prompt v1 / grouped", **metrics_v1,
     "elapsed_seconds": v1_result["elapsed_seconds"], "invalid_outputs": len(v1_result["errors"])},
    {"model": PRIMARY_MODEL_ID, "experiment": "Prompt v2 / grouped", **metrics_v2,
     "elapsed_seconds": v2_result["elapsed_seconds"], "invalid_outputs": len(v2_result["errors"])},
    {"model": PRIMARY_MODEL_ID, "experiment": "Prompt v2 / ordered", **ordered_metrics,
     "elapsed_seconds": ordered_result["elapsed_seconds"], "invalid_outputs": len(ordered_result["errors"])},
]
if optimized_program is not None:
    final_rows.append({"model": PRIMARY_MODEL_ID, "experiment": "DSPy MIPROv2 / grouped", **dspy_group_metrics,
                       "elapsed_seconds": dspy_group_elapsed, "invalid_outputs": len(dspy_group_errors)})
if optimized_ordered_program is not None:
    final_rows.append({"model": PRIMARY_MODEL_ID, "experiment": "DSPy MIPROv2 / ordered", **optimized_ordered_metrics,
                       "elapsed_seconds": optimized_ordered_elapsed, "invalid_outputs": len(optimized_ordered_errors)})
if second_model_id_detected and second_result_grouped is not None:
    final_rows.append({"model": second_model_id_detected, "experiment": "Prompt v2 / grouped", **second_metrics_grouped,
                       "elapsed_seconds": second_result_grouped["elapsed_seconds"], "invalid_outputs": len(second_result_grouped["errors"])})
if second_model_id_detected and second_result_ordered is not None:
    final_rows.append({"model": second_model_id_detected, "experiment": "Prompt v2 / ordered", **second_metrics_ordered,
                       "elapsed_seconds": second_result_ordered["elapsed_seconds"], "invalid_outputs": len(second_result_ordered["errors"])})

final_table = pd.DataFrame(final_rows)
display(final_table)
final_table.to_csv(OUTPUT_DIR / "final_comparison.csv", index=False)
print("Таблица сохранена:", (OUTPUT_DIR / "final_comparison.csv").resolve())

,model,experiment,precision,recall,f1,tp,fp,fn,evaluated_sentences,elapsed_seconds,invalid_outputs
0,/Users/veronika_steklo/Models/qwen2.5-7b-instr...,Prompt v1 / grouped,0.357724,0.571429,0.440000,132,237,99,140,476.204906,0
1,/Users/veronika_steklo/Models/qwen2.5-7b-instr...,Prompt v2 / grouped,0.494505,0.584416,0.535714,135,138,96,140,503.724166,0
2,/Users/veronika_steklo/Models/qwen2.5-7b-instr...,Prompt v2 / ordered,0.554545,0.528139,0.541020,122,98,109,140,421.610046,0
3,/Users/veronika_steklo/Models/qwen2.5-7b-instr...,DSPy MIPROv2 / grouped,0.578704,0.541126,0.559284,125,91,106,140,551.803688,3
4,/Users/veronika_steklo/Models/gemma-4-E2B_q4_0...,Prompt v2 / grouped,0.694805,0.463203,0.555844,107,47,124,140,1557.758294,25
5,/Users/veronika_steklo/Models/gemma-4-E2B_q4_0...,Prompt v2 / ordered,0.683761,0.346320,0.459770,80,37,151,140,1608.722158,36


Таблица сохранена: /Users/veronika_steklo/projects/ml_ssu/task6_local_llm_ollama/work/ner_outputs/final_comparison.csv
